# 02. Сравнение моделей

Обучаем три модели на одном train/test split и сравниваем по метрикам.

**Метрики:**
- **ROC-AUC** — основная (устойчива к дисбалансу классов).
- **F1** — баланс precision/recall.
- **Precision / Recall** — для интерпретации (recall важнее: пропустить уходящего клиента дороже).
- **Accuracy** — справочно.

In [ ]:
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
)
from sklearn.pipeline import Pipeline

from src.data.load import load_clean
from src.data.preprocess import build_preprocessor, train_test_split_stratified
from src.models.train import get_models

sns.set_theme(style='whitegrid')

In [ ]:
df = load_clean(PROJECT_ROOT / 'data' / 'raw' / 'telco_churn.csv')
x_train, x_test, y_train, y_test = train_test_split_stratified(df)
print('Train:', x_train.shape, 'Test:', x_test.shape)

In [ ]:
models = get_models()
fitted = {}
for name, model in models.items():
    pipe = Pipeline([('preprocessor', build_preprocessor()), ('model', model)])
    pipe.fit(x_train, y_train)
    fitted[name] = pipe
print('Готово. Обучено моделей:', len(fitted))

## Сводная таблица метрик

In [ ]:
metrics_path = PROJECT_ROOT / 'artifacts' / 'metrics.json'
if metrics_path.exists():
    summary = json.loads(metrics_path.read_text())
    metrics_df = pd.DataFrame(summary['metrics']).T
    metrics_df = metrics_df[['roc_auc', 'f1', 'precision', 'recall', 'accuracy']]
    print('Best model:', summary['best_model'])
    metrics_df.style.format('{:.4f}').background_gradient(cmap='Greens', axis=0)

## ROC-кривые

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, pipe in fitted.items():
    proba = pipe.predict_proba(x_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc(fpr, tpr):.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax.set_xlabel('FPR')
ax.set_ylabel('TPR')
ax.set_title('ROC-кривые')
ax.legend()
plt.show()

## Confusion matrix для лучшей модели

In [ ]:
best_name = summary['best_model'] if metrics_path.exists() else 'logreg'
best = fitted[best_name]
y_pred = best.predict(x_test)
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stay', 'Churn'], yticklabels=['Stay', 'Churn'], ax=ax)
ax.set_title(f'Confusion matrix — {best_name}')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.show()

print(classification_report(y_test, y_pred, target_names=['Stay', 'Churn']))

## Выводы

По итогам сравнения:

- **LogisticRegression** показала лучший ROC-AUC (~0.84) и наивысший recall (~0.78). Это критично для бизнес-задачи: пропустить уходящего клиента дороже, чем сделать ложноположительное предсказание.
- **RandomForest** и **GradientBoosting** дают сопоставимый ROC-AUC (~0.83), но смещены в сторону precision при низком recall (с balanced weights в LogReg recall выше).
- LogReg также имеет преимущество: **интерпретируемость** (можно посмотреть коэффициенты по фичам), низкая стоимость инференса, простая поддержка.

**Финальная модель:** LogisticRegression с `class_weight='balanced'`. Сохранена в `artifacts/model.pkl` и используется в `/predict` сервиса.